# Wildfire Intensity Classification — Machine Learning Assignment

**Task:** Predict fire intensity level (4 classes: 0=Low, 1=Moderate, 2=High, 3=Extreme) based on environmental and observational features.

**Models implemented:** Decision Tree · Support Vector Machine (SVM) · Neural Network (MLP)

**Evaluation:** Hold-out validation (80/20 split) + k-fold cross-validation for hyperparameter tuning

---

## How to Run
1. Place `wildfire_cls_train_full.csv` and `wildfire_cls_test_features.csv` in the same directory as this notebook.
2. Install dependencies: `pip install pandas numpy scikit-learn matplotlib seaborn`
3. Run all cells top-to-bottom (`Kernel → Restart & Run All`).
4. The final cell saves `test_predictions.csv`.

---

## 0. Imports & Configuration

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')
import random
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Scikit-learn ──────────────────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Models
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Plot style ────────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.15)
FIGSIZE = (10, 5)

CLASS_NAMES = ['Low (0)', 'Moderate (1)', 'High (2)', 'Extreme (3)']

print('All imports successful.')

---
## 1. Load Data

In [ ]:
train_df = pd.read_csv('wildfire_cls_train_full.csv')
test_df  = pd.read_csv('wildfire_cls_test_features.csv')

print(f'Training set : {train_df.shape[0]:,} rows × {train_df.shape[1]} columns')
print(f'Test set     : {test_df.shape[0]:,} rows × {test_df.shape[1]} columns')
train_df.head(3)

In [ ]:
print('\n── Column dtypes ──')
print(train_df.dtypes)
print('\n── Basic statistics ──')
train_df.describe(include='all').T

---
## 2. Exploratory Data Analysis (EDA)

> **What to look for / write in analysis:**
> - Comment on class balance (are classes roughly equal, or skewed?).
> - Note which numeric features have high spread or potential outliers.
> - Note correlations between `brightness_k`, `temp_max_c`, `wind_max_kmh` and the target.
> - Categorical features with high cardinality (e.g. `country`) will need encoding.

In [ ]:
# ── 2.1 Target class distribution ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
vc = train_df['fire_intensity'].value_counts().sort_index()
bars = ax.bar(CLASS_NAMES, vc.values, color=sns.color_palette('muted', 4))
ax.bar_label(bars, fmt='%d')
ax.set_title('Target Class Distribution')
ax.set_xlabel('Fire Intensity Class')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig('fig_class_dist.png', dpi=150)
plt.show()
print(vc)

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# Report whether classes are balanced. If heavily imbalanced, justify using
# macro F1 as the primary metric (it treats all classes equally).

In [ ]:
# ── 2.2 Missing values ────────────────────────────────────────────────────────
missing = train_df.isnull().sum()
missing_pct = (missing / len(train_df) * 100).round(2)
mv = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(mv[mv['Missing Count'] > 0])

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# List which columns have missing values and the %. For numeric columns
# with <5% missing, mean/median imputation is commonly justified. For
# categorical columns, use the mode. Justify your choice here.

In [ ]:
# ── 2.3 Numeric feature distributions ────────────────────────────────────────
num_cols = ['latitude', 'longitude', 'brightness_k',
            'temp_max_c', 'wind_max_kmh', 'precip_mm', 'humidity_pct']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(train_df[col].dropna(), bins=40, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
axes[-1].set_visible(False)
plt.suptitle('Numeric Feature Distributions (Training Set)', y=1.01)
plt.tight_layout()
plt.savefig('fig_num_dist.png', dpi=150)
plt.show()

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# Mention skewed features (e.g. precip_mm often right-skewed).
# SVM and NN are sensitive to feature scale → StandardScaler is essential.

In [ ]:
# ── 2.4 Box-plots: numeric features vs target ─────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    train_df.boxplot(column=col, by='fire_intensity', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('Fire Intensity')
axes[-1].set_visible(False)
plt.suptitle('Feature vs Fire Intensity (Box-plots)')
plt.tight_layout()
plt.savefig('fig_boxplots.png', dpi=150)
plt.show()

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# Discuss which features show clear separation across classes.
# brightness_k should increase with intensity — expected from physics.

In [ ]:
# ── 2.5 Correlation heatmap (numeric only) ─────────────────────────────────────
corr = train_df[num_cols + ['fire_intensity']].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.savefig('fig_corr.png', dpi=150)
plt.show()

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# Identify the top 2-3 features correlated with fire_intensity.
# Note any multi-collinearity (may not affect tree models but affects SVM).

In [ ]:
# ── 2.6 Categorical feature counts ────────────────────────────────────────────
cat_cols = ['season', 'daynight', 'region', 'fire_type', 'satellite',
            'instrument', 'confidence']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    vc2 = train_df[col].value_counts()
    axes[i].bar(vc2.index.astype(str), vc2.values, color='coral')
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=30)
axes[-1].set_visible(False)
plt.suptitle('Categorical Feature Distributions')
plt.tight_layout()
plt.savefig('fig_cat_dist.png', dpi=150)
plt.show()

---
## 3. Pre-processing

> **What to write in analysis:**
> - Justify **why** you drop certain columns (e.g., `acq_date` is a string date; `year`/`month` capture the temporal signal instead).
> - Justify your **imputation strategy** for missing values.
> - Justify **StandardScaler** for SVM/NN (distance- and gradient-based models require normalised inputs).
> - Decision Tree does NOT require scaling, but using the same pipeline keeps comparison fair.

In [ ]:
# ── 3.1 Feature selection ─────────────────────────────────────────────────────
# Columns to drop:
#   acq_date  - raw date string (year & month already extracted)
#   acq_time  - HHMM integer with little predictive value without transformation
# Keep 'country' as ordinal-encoded (35 levels, informative regional signal)

DROP_COLS = ['acq_date', 'acq_time']
TARGET    = 'fire_intensity'

def prepare_features(df, le_dict=None, fit=True):
    """
    Return X (features) and optionally y (target).
    le_dict: dict of LabelEncoders. If fit=True, create & fit; else reuse.
    """
    df = df.copy()
    df.drop(columns=[c for c in DROP_COLS if c in df.columns], inplace=True)

    # ── Encode categorical columns ─────────────────────────────────────────
    cat_features = df.select_dtypes(include='object').columns.tolist()
    if TARGET in cat_features:
        cat_features.remove(TARGET)

    if le_dict is None:
        le_dict = {}

    for col in cat_features:
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            le_dict[col] = le
        else:
            le = le_dict[col]
            # Handle unseen labels gracefully
            df[col] = df[col].astype(str).map(
                lambda x, le=le: le.transform([x])[0]
                if x in le.classes_ else -1
            )

    if TARGET in df.columns:
        y = df.pop(TARGET).values
        return df, y, le_dict
    return df, le_dict


X_raw, y, le_dict = prepare_features(train_df, fit=True)
X_test_raw, _    = prepare_features(test_df, le_dict=le_dict, fit=False)

print(f'Feature matrix shape : {X_raw.shape}')
print(f'Feature names        : {list(X_raw.columns)}')

In [ ]:
# ── 3.2 Impute missing values (numeric) ───────────────────────────────────────
# JUSTIFICATION: Missing values in numeric columns (<5%) are imputed with
# the MEDIAN (robust to outliers). Categorical columns are already encoded;
# rare missing categoricals are filled with mode before encoding above.

imputer = SimpleImputer(strategy='median')
X_imp       = imputer.fit_transform(X_raw)
X_test_imp  = imputer.transform(X_test_raw)

print(f'Any NaN after imputation (train): {np.isnan(X_imp).any()}')
print(f'Any NaN after imputation (test) : {np.isnan(X_test_imp).any()}')

In [ ]:
# ── 3.3 Train / validation split (stratified, 80/20) ──────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X_imp, y, test_size=0.20, random_state=SEED, stratify=y
)
print(f'Training   : {X_train.shape[0]:,} samples')
print(f'Validation : {X_val.shape[0]:,} samples')

# ── 3.4 Scale (for SVM & NN) ──────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test_imp)

print('Scaling done.')

---
## 4. Helper Functions

In [ ]:
def evaluate(model, X_tr, y_tr, X_v, y_v, label=''):
    """Print accuracy & macro F1 for train and validation sets."""
    tr_pred = model.predict(X_tr)
    v_pred  = model.predict(X_v)
    tr_acc  = accuracy_score(y_tr, tr_pred)
    v_acc   = accuracy_score(y_v,  v_pred)
    tr_f1   = f1_score(y_tr, tr_pred, average='macro')
    v_f1    = f1_score(y_v,  v_pred,  average='macro')
    print(f'{label}')
    print(f'  Train  — Accuracy: {tr_acc:.4f}  Macro-F1: {tr_f1:.4f}')
    print(f'  Val    — Accuracy: {v_acc:.4f}  Macro-F1: {v_f1:.4f}')
    return tr_acc, v_acc, tr_f1, v_f1


def plot_confusion(model, X_v, y_v, title='Confusion Matrix'):
    cm = confusion_matrix(y_v, model.predict(X_v))
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(7, 6))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(f'fig_cm_{title.replace(" ","_")}.png', dpi=150)
    plt.show()


def hyperparam_curve(param_vals, tr_scores, val_scores, xlabel, title, log=False):
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.plot(param_vals, tr_scores,  'o-', label='Train')
    ax.plot(param_vals, val_scores, 's--', label='Validation')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Macro F1')
    ax.set_title(title)
    if log:
        ax.set_xscale('log')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'fig_hp_{title.replace(" ","_")}.png', dpi=150)
    plt.show()

print('Helper functions defined.')

---
## 5. Model 1 — Decision Tree

> **What to write in analysis:**
> - A Decision Tree partitions the feature space by choosing splits that maximise information gain (or minimise Gini impurity).
> - `max_depth` is the primary regularisation lever: low depth → underfitting; unrestricted depth → overfitting (perfect train, poor val).
> - Look for the 'elbow' in the validation curve — that is the optimal depth.
> - Feature importance: which features does the tree rely on most? Does this align with the EDA correlation findings?

In [ ]:
# ── 5.1 Hyperparameter 1: max_depth ───────────────────────────────────────────
depths       = [2, 3, 5, 7, 10, 15, 20, None]
depth_labels = [str(d) if d else 'None' for d in depths]
tr_f1s, val_f1s = [], []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=SEED)
    dt.fit(X_train, y_train)
    tr_f1s.append(f1_score(y_train, dt.predict(X_train), average='macro'))
    val_f1s.append(f1_score(y_val,  dt.predict(X_val),  average='macro'))

fig, ax = plt.subplots(figsize=FIGSIZE)
x_pos = range(len(depths))
ax.plot(x_pos, tr_f1s, 'o-', label='Train')
ax.plot(x_pos, val_f1s, 's--', label='Validation')
ax.set_xticks(x_pos)
ax.set_xticklabels(depth_labels)
ax.set_xlabel('max_depth')
ax.set_ylabel('Macro F1')
ax.set_title('Decision Tree — max_depth Hyperparameter Curve')
ax.legend()
plt.tight_layout()
plt.savefig('fig_hp_DT_depth.png', dpi=150)
plt.show()

best_depth = depths[int(np.argmax(val_f1s))]
print(f'Best max_depth (by val Macro-F1): {best_depth}')

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# At depth=None, train F1 ≈ 1.0 and val F1 drops → clear overfitting.
# At depth=2-3, both train and val F1 are low → underfitting.
# Best depth shows the bias-variance tradeoff sweet spot.

In [ ]:
# ── 5.2 Hyperparameter 2: min_samples_split (COSC2793 additional HP) ──────────
min_splits   = [2, 5, 10, 20, 50, 100]
tr_f1s2, val_f1s2 = [], []

for ms in min_splits:
    dt_tmp = DecisionTreeClassifier(
        max_depth=best_depth, min_samples_split=ms, random_state=SEED
    )
    dt_tmp.fit(X_train, y_train)
    tr_f1s2.append(f1_score(y_train, dt_tmp.predict(X_train), average='macro'))
    val_f1s2.append(f1_score(y_val,  dt_tmp.predict(X_val),  average='macro'))

hyperparam_curve(min_splits, tr_f1s2, val_f1s2,
                 'min_samples_split',
                 'Decision Tree — min_samples_split Hyperparameter Curve')

best_mss = min_splits[int(np.argmax(val_f1s2))]
print(f'Best min_samples_split: {best_mss}')

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# min_samples_split prevents splits on very small subsets, acting as an
# additional regulariser on top of max_depth. Larger values → more pruning
# → less overfitting, but potentially more underfitting at very high values.

In [ ]:
# ── 5.3 Final Decision Tree model ─────────────────────────────────────────────
dt_best = DecisionTreeClassifier(
    max_depth=best_depth,
    min_samples_split=best_mss,
    random_state=SEED
)
dt_best.fit(X_train, y_train)

dt_tr_acc, dt_val_acc, dt_tr_f1, dt_val_f1 = evaluate(
    dt_best, X_train, y_train, X_val, y_val, label='Decision Tree (Best)')

print('\nClassification Report (Validation):')
print(classification_report(y_val, dt_best.predict(X_val), target_names=CLASS_NAMES))
plot_confusion(dt_best, X_val, y_val, 'Decision Tree')

In [ ]:
# ── 5.4 Feature importance ────────────────────────────────────────────────────
feat_names = X_raw.columns.tolist()
importances = pd.Series(dt_best.feature_importances_, index=feat_names)
importances = importances.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
importances.plot.bar(ax=ax, color='teal')
ax.set_title('Decision Tree — Feature Importances')
ax.set_ylabel('Importance (Gini)')
plt.tight_layout()
plt.savefig('fig_DT_importance.png', dpi=150)
plt.show()

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# Top features should align with the EDA correlation analysis.
# brightness_k is expected to be the top feature given it is a direct
# thermal signal of fire intensity.

In [ ]:
# ── 5.5 Visualise top 3 levels of the tree ────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 6))
plot_tree(dt_best, max_depth=3, feature_names=feat_names,
          class_names=['Low','Mod','High','Extreme'],
          filled=True, ax=ax, fontsize=8)
ax.set_title('Decision Tree (top 3 levels)')
plt.tight_layout()
plt.savefig('fig_DT_tree.png', dpi=150)
plt.show()

---
## 6. Model 2 — Support Vector Machine (SVM)

> **What to write in analysis:**
> - SVM finds a maximum-margin hyperplane. With RBF kernel it can handle non-linear boundaries.
> - `C` controls the trade-off between margin width and misclassification penalty: low C → soft margin (regularised); high C → hard margin (may overfit).
> - `gamma` ('scale' default) controls the influence radius of each training point.
> - SVM requires **feature scaling** — without it, features on different scales dominate the margin.
> - SVM is computationally expensive on large datasets; note any time constraints.

In [ ]:
# ── 6.1 Hyperparameter 1: C (regularisation) ──────────────────────────────────
C_values = [0.01, 0.1, 1, 10, 100]
tr_f1s_svm, val_f1s_svm = [], []

for C in C_values:
    svm_tmp = SVC(C=C, kernel='rbf', gamma='scale', random_state=SEED)
    svm_tmp.fit(X_train_sc, y_train)
    tr_f1s_svm.append(f1_score(y_train, svm_tmp.predict(X_train_sc), average='macro'))
    val_f1s_svm.append(f1_score(y_val,  svm_tmp.predict(X_val_sc),  average='macro'))

hyperparam_curve(C_values, tr_f1s_svm, val_f1s_svm,
                 'C (regularisation)', 'SVM — C Hyperparameter Curve', log=True)

best_C = C_values[int(np.argmax(val_f1s_svm))]
print(f'Best C: {best_C}')

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# At very low C, the model under-penalises misclassifications → underfitting.
# At very high C, the model may overfit the training data.
# The best C balances train/val performance.

In [ ]:
# ── 6.2 Hyperparameter 2: gamma (COSC2793) ────────────────────────────────────
gamma_vals = [0.001, 0.01, 0.1, 1.0, 'scale']
tr_f1s_g, val_f1s_g = [], []

for g in gamma_vals:
    svm_g = SVC(C=best_C, kernel='rbf', gamma=g, random_state=SEED)
    svm_g.fit(X_train_sc, y_train)
    tr_f1s_g.append(f1_score(y_train, svm_g.predict(X_train_sc), average='macro'))
    val_f1s_g.append(f1_score(y_val,  svm_g.predict(X_val_sc),  average='macro'))

x_pos = range(len(gamma_vals))
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(x_pos, tr_f1s_g,  'o-', label='Train')
ax.plot(x_pos, val_f1s_g, 's--', label='Validation')
ax.set_xticks(x_pos)
ax.set_xticklabels([str(g) for g in gamma_vals])
ax.set_xlabel('gamma')
ax.set_ylabel('Macro F1')
ax.set_title('SVM — gamma Hyperparameter Curve')
ax.legend()
plt.tight_layout()
plt.savefig('fig_hp_SVM_gamma.png', dpi=150)
plt.show()

best_gamma = gamma_vals[int(np.argmax(val_f1s_g))]
print(f'Best gamma: {best_gamma}')

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# High gamma → very localised decision boundary → overfitting.
# Low gamma → very smooth boundary → underfitting.
# 'scale' (= 1/(n_features * X.var())) is a robust default.

In [ ]:
# ── 6.3 Final SVM model ───────────────────────────────────────────────────────
svm_best = SVC(C=best_C, kernel='rbf', gamma=best_gamma, random_state=SEED)
svm_best.fit(X_train_sc, y_train)

svm_tr_acc, svm_val_acc, svm_tr_f1, svm_val_f1 = evaluate(
    svm_best, X_train_sc, y_train, X_val_sc, y_val, label='SVM (Best)')

print('\nClassification Report (Validation):')
print(classification_report(y_val, svm_best.predict(X_val_sc), target_names=CLASS_NAMES))
plot_confusion(svm_best, X_val_sc, y_val, 'SVM')

---
## 7. Model 3 — Neural Network (MLP)

> **What to write in analysis:**
> - MLP learns hierarchical feature representations via layers of neurons with non-linear activations.
> - `learning_rate_init` controls step size during gradient descent. Too high → divergence/oscillation; too low → slow convergence.
> - The training **loss curve** should decrease and plateau; a widening gap between train and validation loss signals overfitting.
> - `hidden_layer_sizes` defines network capacity. More layers/neurons → more capacity (can overfit on small data).
> - Use `loss_curve_` attribute to plot training loss across epochs.

In [ ]:
# ── 7.1 Hyperparameter 1: learning_rate_init ──────────────────────────────────
lr_vals = [0.0001, 0.001, 0.01, 0.05, 0.1]
tr_f1s_nn, val_f1s_nn = [], []
loss_curves = {}

for lr in lr_vals:
    mlp_tmp = MLPClassifier(
        hidden_layer_sizes=(128, 64),
        learning_rate_init=lr,
        max_iter=200,
        random_state=SEED,
        early_stopping=False,
        solver='adam'
    )
    mlp_tmp.fit(X_train_sc, y_train)
    tr_f1s_nn.append(f1_score(y_train, mlp_tmp.predict(X_train_sc), average='macro'))
    val_f1s_nn.append(f1_score(y_val,  mlp_tmp.predict(X_val_sc),  average='macro'))
    loss_curves[lr] = mlp_tmp.loss_curve_

hyperparam_curve(lr_vals, tr_f1s_nn, val_f1s_nn,
                 'learning_rate_init', 'Neural Network — Learning Rate Curve', log=True)

best_lr = lr_vals[int(np.argmax(val_f1s_nn))]
print(f'Best learning_rate_init: {best_lr}')

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# At 0.1 the training may oscillate/diverge (loss_curve_ is jagged).
# At 0.0001 the model may not converge in 200 epochs (loss still decreasing).
# Optimal LR (often 0.001 with Adam) converges smoothly.

In [ ]:
# ── 7.2 Training loss curves for each learning rate ───────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
for lr, curve in loss_curves.items():
    ax.plot(curve, label=f'LR={lr}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Neural Network — Training Loss Curves by Learning Rate')
ax.legend()
plt.tight_layout()
plt.savefig('fig_NN_loss_curves.png', dpi=150)
plt.show()

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# Compare convergence speed and stability across learning rates.
# A smooth, monotonically decreasing curve indicates stable training.
# Noisy or non-decreasing curves suggest the LR is too high.

In [ ]:
# ── 7.3 Hyperparameter 2: hidden_layer_sizes (COSC2793) ───────────────────────
architectures = [(32,), (64, 32), (128, 64), (256, 128, 64), (64, 64, 64)]
arch_labels   = [str(a) for a in architectures]
tr_f1s_arch, val_f1s_arch = [], []

for arch in architectures:
    mlp_a = MLPClassifier(
        hidden_layer_sizes=arch,
        learning_rate_init=best_lr,
        max_iter=200,
        random_state=SEED,
        solver='adam'
    )
    mlp_a.fit(X_train_sc, y_train)
    tr_f1s_arch.append(f1_score(y_train, mlp_a.predict(X_train_sc), average='macro'))
    val_f1s_arch.append(f1_score(y_val,  mlp_a.predict(X_val_sc),  average='macro'))

x_pos = range(len(architectures))
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_pos, tr_f1s_arch,  'o-', label='Train')
ax.plot(x_pos, val_f1s_arch, 's--', label='Validation')
ax.set_xticks(x_pos)
ax.set_xticklabels(arch_labels, rotation=20)
ax.set_xlabel('Architecture (hidden layers)')
ax.set_ylabel('Macro F1')
ax.set_title('Neural Network — Architecture Hyperparameter Curve')
ax.legend()
plt.tight_layout()
plt.savefig('fig_hp_NN_arch.png', dpi=150)
plt.show()

best_arch = architectures[int(np.argmax(val_f1s_arch))]
print(f'Best architecture: {best_arch}')

# ANALYSIS NOTE ─────────────────────────────────────────────────────────────
# Very small networks underfit; very large networks overfit on this dataset.
# Discuss the bias-variance tradeoff in terms of model capacity.

In [ ]:
# ── 7.4 Final Neural Network model ───────────────────────────────────────────
nn_best = MLPClassifier(
    hidden_layer_sizes=best_arch,
    learning_rate_init=best_lr,
    max_iter=500,
    random_state=SEED,
    solver='adam',
    early_stopping=False
)
nn_best.fit(X_train_sc, y_train)

nn_tr_acc, nn_val_acc, nn_tr_f1, nn_val_f1 = evaluate(
    nn_best, X_train_sc, y_train, X_val_sc, y_val, label='Neural Network (Best)')

print('\nClassification Report (Validation):')
print(classification_report(y_val, nn_best.predict(X_val_sc), target_names=CLASS_NAMES))
plot_confusion(nn_best, X_val_sc, y_val, 'Neural Network')

# ── Final training loss curve ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.plot(nn_best.loss_curve_, color='steelblue')
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss')
ax.set_title('Neural Network (Best) — Training Loss Curve')
plt.tight_layout()
plt.savefig('fig_NN_best_loss.png', dpi=150)
plt.show()

---
## 8. Cross-Validation on Best Models

> **What to write in analysis:**
> - Cross-validation gives a more robust estimate of generalisation than a single validation split.
> - Report mean ± std of CV scores. High std → high variance (model is sensitive to which data it sees).
> - Compare CV scores to your hold-out validation scores to check consistency.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# DT — no scaling needed
dt_cv = cross_val_score(dt_best, X_imp, y, cv=skf, scoring='f1_macro', n_jobs=-1)

# SVM & NN — use scaled data (refit scaler inside CV for correctness)
from sklearn.pipeline import make_pipeline

svm_pipe = make_pipeline(StandardScaler(),
    SVC(C=best_C, kernel='rbf', gamma=best_gamma, random_state=SEED))
svm_cv = cross_val_score(svm_pipe, X_imp, y, cv=skf, scoring='f1_macro', n_jobs=-1)

nn_pipe = make_pipeline(StandardScaler(),
    MLPClassifier(hidden_layer_sizes=best_arch, learning_rate_init=best_lr,
                  max_iter=500, random_state=SEED, solver='adam'))
nn_cv = cross_val_score(nn_pipe, X_imp, y, cv=skf, scoring='f1_macro', n_jobs=-1)

print('5-Fold Cross-Validation (Macro F1):')
print(f'  Decision Tree : {dt_cv.mean():.4f} ± {dt_cv.std():.4f}')
print(f'  SVM           : {svm_cv.mean():.4f} ± {svm_cv.std():.4f}')
print(f'  Neural Network: {nn_cv.mean():.4f} ± {nn_cv.std():.4f}')

---
## 9. Model Comparison & Final Selection

> **What to write in analysis:**
> - Compare all three models on Accuracy, Macro-F1 (validation), and CV Macro-F1.
> - The best model should show: high val F1, small train-val gap (low overfitting), consistent CV scores.
> - Justify your final model choice with specific numbers.
> - Mention practical factors: SVM is slow to train; NN needs scaling and more tuning; DT is interpretable.

In [ ]:
results = pd.DataFrame({
    'Model'       : ['Decision Tree', 'SVM', 'Neural Network'],
    'Train Acc'   : [dt_tr_acc,  svm_tr_acc,  nn_tr_acc],
    'Val Acc'     : [dt_val_acc, svm_val_acc, nn_val_acc],
    'Train Macro-F1': [dt_tr_f1,  svm_tr_f1,  nn_tr_f1],
    'Val Macro-F1': [dt_val_f1,  svm_val_f1,  nn_val_f1],
    'CV Macro-F1 mean': [dt_cv.mean(), svm_cv.mean(), nn_cv.mean()],
    'CV Macro-F1 std' : [dt_cv.std(),  svm_cv.std(),  nn_cv.std()],
})
print(results.to_string(index=False))

# ── Bar chart comparison ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(3)
w = 0.25
models = ['Decision Tree', 'SVM', 'Neural Network']
ax.bar(x - w,   results['Val Acc'],      w, label='Val Accuracy')
ax.bar(x,       results['Val Macro-F1'], w, label='Val Macro-F1')
ax.bar(x + w,   results['CV Macro-F1 mean'], w, label='CV Macro-F1')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_title('Model Comparison')
ax.legend()
plt.tight_layout()
plt.savefig('fig_model_comparison.png', dpi=150)
plt.show()

In [ ]:
# ── Select final model ────────────────────────────────────────────────────────
# UPDATE: change `final_model`, `final_X_train`, `final_X_val`, `final_X_test`
# to whichever model achieved the best CV Macro-F1.

# *** CHANGE THIS to 'dt', 'svm', or 'nn' based on your results ***
best_model_name = 'nn'   # example: 'nn'

if best_model_name == 'dt':
    final_model   = dt_best
    final_X_train = X_train
    final_X_val   = X_val
    final_X_test  = X_test_imp
elif best_model_name == 'svm':
    final_model   = svm_best
    final_X_train = X_train_sc
    final_X_val   = X_val_sc
    final_X_test  = X_test_sc
else:  # nn
    final_model   = nn_best
    final_X_train = X_train_sc
    final_X_val   = X_val_sc
    final_X_test  = X_test_sc

print(f'Final model selected: {best_model_name.upper()}')
evaluate(final_model, final_X_train, y_train, final_X_val, y_val, label='FINAL MODEL')
plot_confusion(final_model, final_X_val, y_val, 'Final Model')

---
## 10. Generate Test Predictions

In [ ]:
test_preds = final_model.predict(final_X_test)

submission = pd.DataFrame({
    'id': range(len(test_preds)),
    'fire_intensity': test_preds
})
submission.to_csv('test_predictions.csv', index=False)
print(f'Saved test_predictions.csv  ({len(test_preds)} rows)')
submission.head(10)

---
## 11. Discussion — Limitations, Ethics & Professional Responsibilities

> **Expand this section in your written report. Key points to address:**
>
> ### Limitations
> - The model is only validated on the 7 regions / 35 countries in the training set. Generalisation to unseen geographies is not guaranteed.
> - `brightness_k` is likely the dominant feature; the model may degrade if satellite sensor calibration changes.
> - Class boundaries (Low/Moderate/High/Extreme) are categorical labels derived from continuous brightness temperatures. Label noise near decision boundaries is likely.
> - Missing weather data (`temp_max_c`, `wind_max_kmh`, `precip_mm`) was imputed with medians — actual conditions at fire time may differ.
>
> ### Ethical Considerations
> - **Underestimation risk:** Predicting Low/Moderate when a fire is actually Extreme could delay evacuation orders and cost lives. A cost-sensitive approach (penalising Extreme misclassifications more) should be considered in deployment.
> - **Overestimation risk:** Predicting Extreme when a fire is Low could cause unnecessary resource allocation and public alarm.
> - **Fairness:** Model accuracy may vary by region. Decision-makers should audit per-region performance, especially in regions underrepresented in training data.
> - **Transparency:** Stakeholders (fire authorities, governments) should be informed of model uncertainty and that predictions should be used alongside human judgment.
>
> ### Professional Responsibilities
> - Document model limitations in any deployment setting.
> - Re-validate the model regularly as new fire seasons produce new data patterns.
> - Ensure predictions are not used as the sole basis for life-safety decisions.
> - Comply with open data licensing: the source dataset is CC0 but the assignment-specific processed data must not be distributed.

---
## 12. Summary Table

| Model | Val Accuracy | Val Macro-F1 | CV Macro-F1 | Overfitting? |
|---|---|---|---|---|
| Decision Tree | *fill in* | *fill in* | *fill in* | *comment* |
| SVM | *fill in* | *fill in* | *fill in* | *comment* |
| Neural Network | *fill in* | *fill in* | *fill in* | *comment* |

**Selected model:** *state name and justify with 2-3 sentences referencing your results.*

---
*End of notebook.*